# Detecção de Profissões
O objetivo do exercício é criar um classificador capaz de reconhecer a profissão do solicitante com base na requisição recebida. Utiliza-se um conjunto de dados o qual contém o conteúdo da requisição e a profissão do emissor. As categorias consideradas pela classe alvo são: **governament**, **private** e **academic**

In [1]:
# importação de bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Importando conjunto de dados ***"ep2-train.csv"***

In [2]:
path_file = "../ep2-train.csv"
df = pd.read_csv(path_file, encoding='latin1', sep=';')

In [3]:
df.head()

,req_text,profession
0,Sou aluna de doutorado da UFPR e pesquiso sobr...,academic
1,Gostaria de consultar a disponibilidade de cód...,academic
2,"Prezados, bom dia. Procurei no site do Serpro ...",government
3,Solicito o número de matrícula do SIAP do doce...,academic
4,A Lei nº 12772/2012 regulamenta a carreira de ...,academic


In [4]:
df['profession'].value_counts()

profession
government    18782
academic      14593
private       10303
Name: count, dtype: int64

Como é possível visualizar, ***"ep2-train.csv"*** é um conjunto desbalanceado, contendo majoritariamente requisições de pessoas do **governo**. Logo, a acurácia não é uma métrica confiável para avaliar o desempenho do classificador.

In [5]:
df.isnull().sum()

req_text      0
profession    0
dtype: int64

In [6]:
df.duplicated().sum()

6898

O conjunto possui algumas instâncias **repetidas**, mas não há dados **faltantes**. Pode haver requisições iguais com profissões distintas, porém isso será mantido, visto que não como decidir qual é a profissão correta e uma exclusão simples poderia acarretar em uma perda **significativa** de dados. Logo, será retirado apenas as instâncias repetidas considerado todas as **colunas**.

In [7]:
df = df.drop_duplicates() # retira instâncias duplicadas

In [8]:
df.duplicated().sum()

0

Será aplicado uma transformação **Label Encoding** sobre ***"profession"***, por se tratar de uma coluna categórica.

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['profession_encoded'] = le.fit_transform(df['profession']) #transformação númerica simples

In [10]:
df.head()

,req_text,profession,profession_encoded
0,Sou aluna de doutorado da UFPR e pesquiso sobr...,academic,0
1,Gostaria de consultar a disponibilidade de cód...,academic,0
2,"Prezados, bom dia. Procurei no site do Serpro ...",government,1
3,Solicito o número de matrícula do SIAP do doce...,academic,0
4,A Lei nº 12772/2012 regulamenta a carreira de ...,academic,0


## Modelagem da Representação Textual

Com base em experiências passadas, será apenas explorado o impacto da ***representação textual***, pois se viu que o algoritmo utilizado pouco influencia o desempenho. O desempenho é determinado **majoritamente** pela representação utilizada.

In [11]:
#Bag-of-words
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
X = vect.fit_transform(df['req_text']) # BoW com frequência normalizada

In [12]:
X.shape

(36780, 67761)

Foi gerado um Bag-of-Words com um vocabulário de 67.761 palavras (tokens).

Como é um problema desbalanceado e se deseja verificar o desempenho geral, será utilizado a métrica ***F1-Score Weighted***.

In [13]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
results = cross_validate(
    estimator=model,
    X = X,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score=False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [14]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.593320,0.587717
1,0.590686,0.585200
2,0.577307,0.570975
3,0.578836,0.573888
4,0.589538,0.584905


Como é possível perceber, o uso do BoW não gerou resultados espetáculares. Agora, iremos explorar BoW com TF-IDF

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer()
X_tf = tf_idf.fit_transform(df['req_text']) #BoW TF-IDF

In [16]:
X_tf.shape

(36780, 67761)

BoW TF-IDF gerou a mesma quantidade de atributos.

In [17]:
results = cross_validate(
    estimator=model,
    X = X_tf,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [18]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.623818,0.619135
1,0.622636,0.617805
2,0.609012,0.603855
3,0.612284,0.608147
4,0.615938,0.611320


Fazer uso do **TF-IDF** melhorou relativamente o desempenho do classificador. Com base nisso, iremos explorar o uso de n-gramas com TF-IDF. Iremos testar os seguintes intervalos de n_gramas:
- (1, 2): unigramas, bigramas
- (1, 3): unigramas, bigramas e trigramas

In [19]:
tf_gr1 = TfidfVectorizer(ngram_range=(1,2)) # unigramas e bigramas
X_gr1 = tf_gr1.fit_transform(df['req_text'])
X_gr1.shape

(36780, 724726)

In [20]:
results = cross_validate(
    estimator=model,
    X = X_gr1,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [21]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.640742,0.635836
1,0.644275,0.639312
2,0.630485,0.625025
3,0.635907,0.630908
4,0.640335,0.635378


In [22]:
tf_gr2 = TfidfVectorizer(ngram_range=(1,3)) # Unigramas, bigramas e trigramas
X_gr2 = tf_gr2.fit_transform(df['req_text'])
X_gr2.shape

(36780, 2079098)

In [23]:
results = cross_validate(
    estimator=model,
    X = X_gr2,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [24]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.642500,0.637157
1,0.643654,0.638568
2,0.632219,0.627105
3,0.635304,0.630073
4,0.642005,0.636465


Houve uma pequena melhora no desempenho ao usar ***n-gramas***. Entretanto, ambos os intervalos testados demonstraram a mesma capacidade preditiva. Logo, será adotado **Unigramas+Bigramas** por conta da dimensionalidade, o qual é menor. Devido ao fenômeno da **maldição da dimensionalidade**, será aplicado uma redução de dimensionalidade.

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ("select", SelectKBest(score_func=chi2)),
    ("clf", LogisticRegression(random_state=42))
])

In [26]:
param_grid = {
    "select__k": [250000, 350000, 450000, 550000, 650000, 700000] #grade de busca
}

In [27]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1_weighted",
    n_jobs=-1
)

In [28]:
grid.fit(X_gr1, df['profession_encoded'])

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('select',
                                        SelectKBest(score_func=<function chi2 at 0x0000019392F33420>)),
                                       ('clf',
                                        LogisticRegression(random_state=42))]),
             n_jobs=-1,
             param_grid={'select__k': [250000, 350000, 450000, 550000, 650000,
                                       700000]},
             scoring='f1_weighted')

In [29]:
grid.best_params_

{'select__k': 650000}

In [30]:
grid.best_score_

0.6369902793983157

Conforme o observado, conseguiu-se manter o desempenho original reduzindo o número de features, diminuindo os efeitos da maldição da dimensionalidade. Logo, para a construção do classificador, será usado ***K=650.000***.

Devido aos resultados obtidos durante o EP anterior, será utilizado o algoritmo **Multinomial Naive Bayes**.

In [31]:
selector = SelectKBest(score_func=chi2, k=650000) # redução de dimensionalidade
X_new = selector.fit_transform(X_gr1, df['profession_encoded'])

In [32]:
from sklearn.naive_bayes import MultinomialNB
model_nb = MultinomialNB()

In [33]:
results = cross_validate(
    estimator=model_nb,
    X = X_new,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [34]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.515244,0.490226
1,0.526226,0.500898
2,0.511895,0.486614
3,0.527224,0.502618
4,0.531390,0.506520


O algoritmo Multinominal Naive Bayes obteve um desempenho pior à Regressão Logística.

In [ ]:
from sklearn.svm import SVC
model_svc = SVC(kernel='linear')
results = cross_validate(
    estimator=model_svc,
    X = X_new,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

Sabendo que há uma boa distinção entre o estilo de escrita das profissões abordadas, talvez haja uma melhora no desempenho do classificador ao usar caracterização autoral. No caso, como há somente a requisição, será extraído características relacionadas ao estilo de escrita.

In [35]:
def extract_author_style(text): #extraí características linguísticas
    words = text.split()
    return {
        'word_count': len(words),
        'avg_word_len': np.mean([len(w) for w in words]) if words else 0,
        'sentece_count': text.count('.'),
        'comma_count': text.count(','),
        'uppercase_ratio': sum(c.isupper() for c in text) / len(text) if text else 0,
        'unique_ratio': len(set(words)) / len(words) if words else 0
    }

In [36]:
df_features = df['req_text'].apply(lambda t: pd.Series(extract_author_style(t))) # features linguísticas

In [37]:
from scipy.sparse import hstack

X_combined = hstack([X_new, df_features.values]) # combina vocabulário com estilo de escrita

In [38]:
results = cross_validate( #features com Regressão Logística
    estimator=model,
    X = X_combined,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [39]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.537604,0.529803
1,0.530220,0.522088
2,0.515396,0.507779
3,0.542094,0.536410
4,0.507184,0.499511


In [40]:
results = cross_validate( #features com Regressão Logística
    estimator=model,
    X = df_features,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [41]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.268833,0.237307
1,0.270084,0.238934
2,0.280685,0.249398
3,0.282506,0.250482
4,0.278069,0.246715


In [42]:
df_features.head()

,word_count,avg_word_len,sentece_count,comma_count,uppercase_ratio,unique_ratio
0,44.0,5.477273,5.0,0.0,0.042254,0.840909
1,20.0,5.300000,2.0,1.0,0.064000,0.850000
2,56.0,7.803571,10.0,3.0,0.016260,0.839286
3,21.0,4.952381,1.0,1.0,0.104839,0.857143
4,83.0,5.445783,5.0,5.0,0.050562,0.843373


Como foi constatado, o uso das ***features linguísticas*** piorou o desempenho do classificador. Logo, a única alternativa para obter melhores resultados é aplicar algoritmos baseados na arquitetura **transformers**.

Entretanto, testaremos antes o uso de ***word embeddings estáticos***. No caso, treinaremos um modelo próprio de **Word2Vec**, pois essa abordagem tende a capturar melhor **nuancias específicas** do problema. Usaremos o algoritmo de treinamento **Skip-Gram**.

In [43]:
from gensim.models import Word2Vec

sentences = [text.split() for text in df['req_text']]

In [44]:
w2v_model = Word2Vec(
    sentences,
    vector_size=300, #tamanho embedding
    window=5,
    min_count=2, 
    sg=1, # Skig-gram
    workers=4,
    epochs=10
)

KeyboardInterrupt: 

In [49]:
w2v_model.save("word2vec.model") #salva o modelo de embeddings estáticos

In [50]:
def text_to_vector(text): #transforma a entrada em embeddings
    words = text.split()
    word_vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if len(word_vecs) == 0:
        return np.zeros(w2v_model.wv.vector_size)
    return np.mean(word_vecs, axis=0) #agregação por média

In [51]:
X_embeddings = np.array([text_to_vector(t) for t in df['req_text']])

In [52]:
results = cross_validate( 
    estimator=model,
    X = X_embeddings,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [53]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.558494,0.552617
1,0.560669,0.555464
2,0.552186,0.546039
3,0.557605,0.551306
4,0.548989,0.542853


# Modelo Final

In [38]:
results = cross_validate( 
    estimator=model,
    X = X_new,
    y = df['profession_encoded'],
    cv = 10,
    scoring = ['f1_weighted', 'f1_macro', 'accuracy'],
    return_train_score= False
)

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

In [39]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df.mean()

test_f1_weighted    0.653545
test_f1_macro       0.648692
test_accuracy       0.654296
dtype: float64

In [40]:
model.fit(X_new, df['profession_encoded']) #Treina modelo final

LogisticRegression(random_state=42)

In [42]:
#Pegando o nome das features utilizadas pelo modelo
feature_names = np.array(tf_gr1.get_feature_names_out())

mask = selector.get_support()

selected_features = feature_names[mask]

In [44]:
import eli5
eli5.show_weights(model, top=20, feature_names=selected_features, target_names=le.classes_) #Interpreta as top 20 features mais importantes